In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bagavathypriya/spam-ham-dataset")

print("Path to dataset files:", path)

100%|██████████| 207k/207k [00:00<00:00, 52.8MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/bagavathypriya/spam-ham-dataset/versions/1


In [ ]:
import os
import pandas as pd
print(os.listdir(path))


['spamhamdata.csv']


In [ ]:
message = pd.read_csv(os.path.join(path, "spamhamdata.csv"), sep='\t', names=['label', 'message'])
message.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
ps=PorterStemmer()



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
corpus=[]
for i in range(0,len(message)):
  review=re.sub('[^a-zA-Z]'," ",message['message'][i])
  review=review.lower()
  review=review.split()
  review=[ps.stem(word) for word in review if word not in stopwords.words('english')]
  review=" ".join(review)
  corpus.append(review)

In [ ]:
#vectorizing
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=2500)
x=cv.fit_transform(corpus).toarray()
y=pd.get_dummies(message["label"]).astype(int)
y

,ham,spam
0,1,0
1,1,0
2,0,1
3,1,0
4,1,0
...,...,...
5567,0,1
5568,1,0
5569,1,0
5570,1,0


In [ ]:
o#ham clafficati
y=y.iloc[:,0]
y

,ham
0,1
1,1
2,0
3,1
4,1
...,...
5567,0
5568,1
5569,1
5570,1


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.20)


In [ ]:
from sklearn.naive_bayes import MultinomialNB
spam_detection=MultinomialNB().fit(X_train,y_train)
y_pred=spam_detection.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9856502242152466
              precision    recall  f1-score   support

           0       0.95      0.94      0.95       148
           1       0.99      0.99      0.99       967

    accuracy                           0.99      1115
   macro avg       0.97      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [ ]:
#creating tfidf model
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=2500,ngram_range=(1,1))
x1=tfidf.fit_transform(corpus).toarray()
from sklearn.model_selection import train_test_split
X1_train,X1_test,y1_train,y1_test=train_test_split(x1,y,test_size=0.20)
spam_detection.fit(X1_train,y1_train)
y1_pred=spam_detection.predict(X1_test)
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y1_test,y1_pred))
print(classification_report(y1_test,y1_pred))

0.9802690582959641
              precision    recall  f1-score   support

           0       0.98      0.86      0.91       137
           1       0.98      1.00      0.99       978

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
# Testing the spam detection model

message = ["Congratulations! You have won a free iPhone. Click here to claim your prize!"]

# Transform the message using the trained TF-IDF vectorizer
message_tfidf = tfidf.transform(message).toarray()

# Predict
prediction = spam_detection.predict(message_tfidf)

# Display result
if prediction[0] == 0:
    print("🚨 Spam Message")
else:
    print("✅ Not Spam (Ham)")

🚨 Spam Message


In [ ]:
# Sample SMS messages

messages = [
    "Congratulations! You won a free prize. Call now!",
    "Hey bro, are you coming to college today?",
    "URGENT! You have won a cash reward. Claim now!",
    "Can you send me the class notes?"
]

# Convert messages to TF-IDF features
messages_tfidf = tfidf.transform(messages).toarray()

# Predict
predictions = spam_detection.predict(messages_tfidf)

# Display predictions
for message, prediction in zip(messages, predictions):
    label = "Spam" if prediction == 0 else "Ham"

    print(f"Message: {message}")
    print(f"Prediction: {label}")
    print("-" * 60)


Message: Congratulations! You won a free prize. Call now!
Prediction: Spam
------------------------------------------------------------
Message: Hey bro, are you coming to college today?
Prediction: Ham
------------------------------------------------------------
Message: URGENT! You have won a cash reward. Claim now!
Prediction: Spam
------------------------------------------------------------
Message: Can you send me the class notes?
Prediction: Ham
------------------------------------------------------------


In [ ]:
message = ["Congratulations! You have won a free prize!"]

message_tfidf = tfidf.transform(message).toarray()

prediction = spam_detection.predict(message_tfidf)[0]
probabilities = spam_detection.predict_proba(message_tfidf)[0]

print("Message:", message[0])
print("Prediction:", "Spam" if prediction == 0 else "Ham")
print("Class labels:", spam_detection.classes_)
print("Prediction probabilities:", probabilities)

Message: Congratulations! You have won a free prize!
Prediction: Spam
Class labels: [0 1]
Prediction probabilities: [0.91304658 0.08695342]
